# Earthquake Magnitude v5 — Data Collection
India / Himalayan belt only. Downloads waveforms from IRIS, labels phases with STA/LTA, saves CSV + `_meta.json` pairs to `processed_v5/`.

In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIG                                        ║
# ╚══════════════════════════════════════════════════════════╝

WAVEFORM_DIR   = 'waveforms_v5'
PROCESSED_DIR  = 'processed_v5'
CATALOG_CSV    = 'catalog_v5_expanded.csv'   # new file — expanded region

# ── Targets ─────────────────────────────────────────────────────────────────
# We use ALL events found in the catalog (no random sampling).
# These are just caps — download stops early if catalog is exhausted.
NUM_TRAIN      = 40000
NUM_TEST       = 5000
MIN_MAGNITUDE  = 3.0    # M3.0+ — picks up more events; IRIS archives these reliably

# ── Date range ───────────────────────────────────────────────────────────────
TRAIN_START    = '2000-02-01'
TRAIN_END      = '2020-02-01'
TEST_START     = '2020-02-01'
TEST_END       = '2024-01-01'    # extended to 2024

# ── Waveform window ──────────────────────────────────────────────────────────
PRE_EVENT_SEC  = 30
POST_EVENT_SEC = 120    # 150s total — P-wave always arrives within this window

# ── Signal processing ────────────────────────────────────────────────────────
FREQMIN        = 1.0
FREQMAX        = 10.0   # keep in sync with TARGET_SR=20 Hz in training
STA_SEC        = 1.0
LTA_SEC        = 10.0
THR_ON         = 2.5
THR_OFF        = 1.0
MIN_P_FRACTION = 0.01

LOC_NORM = [20000.0, 700.0, 90.0, 180.0]

# ── Multi-region search boxes ────────────────────────────────────────────────
# Query all five at once — gives 5–10× more events vs India-only
SEARCH_REGIONS = [
    # name,               minlat, maxlat, minlon, maxlon
    ('India/Himalaya',         5,     40,     60,    100),
    ('Southeast_Asia',       -12,     25,     90,    135),  # Indonesia, Philippines, Myanmar
    ('Central_Asia/Zagros',   25,     50,     40,     80),  # Iran, Turkey, Pakistan, Afghanistan
    ('Japan/Korea',           25,     50,    125,    150),  # Japan Sea + Pacific margin
    ('China/Tibet',           20,     50,     75,    125),  # Tibet, Sichuan, Yunnan
]

import os, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

for d in [WAVEFORM_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config loaded.')
print(f'  MIN_MAGNITUDE  : {MIN_MAGNITUDE}')
print(f'  NUM_TRAIN cap  : {NUM_TRAIN:,}')
print(f'  NUM_TEST  cap  : {NUM_TEST:,}')
print(f'  Train period   : {TRAIN_START} → {TRAIN_END}')
print(f'  Test  period   : {TEST_START} → {TEST_END}')
print(f'  Search regions : {len(SEARCH_REGIONS)}')
for name, la1, la2, lo1, lo2 in SEARCH_REGIONS:
    print(f'    {name:<22} lat [{la1:>+4},{la2:>+4}]  lon [{lo1:>+4},{lo2:>+4}]')

Config loaded.
  MIN_MAGNITUDE  : 3.0
  NUM_TRAIN cap  : 40,000
  NUM_TEST  cap  : 5,000
  Train period   : 2000-02-01 → 2020-02-01
  Test  period   : 2020-02-01 → 2024-01-01
  Search regions : 5
    India/Himalaya         lat [  +5, +40]  lon [ +60,+100]
    Southeast_Asia         lat [ -12, +25]  lon [ +90,+135]
    Central_Asia/Zagros    lat [ +25, +50]  lon [ +40, +80]
    Japan/Korea            lat [ +25, +50]  lon [+125,+150]
    China/Tibet            lat [ +20, +50]  lon [ +75,+125]


## Cell 2 — Install & Imports

In [2]:
# CELL 2 — INSTALL & IMPORTS
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'obspy', 'tqdm', 'numpy', 'pandas', 'requests'], check=False)

import io, math, threading, time, shutil
import numpy as np
import pandas as pd
import requests
import obspy
from obspy.clients.fdsn import Client
from obspy.signal.trigger import recursive_sta_lta, trigger_onset
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

print(f'ObsPy  {obspy.__version__}')
print(f'NumPy  {np.__version__}')
print(f'Pandas {pd.__version__}')



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: /Users/upendrasuryajonnalagadda/earthd/.venv/bin/python -m pip install --upgrade pip


ObsPy  1.5.0
NumPy  1.26.4
Pandas 2.3.3


## Cell 3 — Fetch Catalog (India/Himalayan region)
Queries USGS for M3.5+ events bounded to lat 6–37, lon 68–97. All events kept in chronological order — no random sampling.

In [3]:
# CELL 3 — FETCH CATALOG FROM USGS (multi-region)
# Queries 5 overlapping boxes covering Asia + Western Pacific.
# Deduplicates events that appear in multiple boxes (same event_id).
# Splits into train / test by date. Saves to catalog_v5_expanded.csv.

import io, requests
import numpy as np
import pandas as pd

USGS_URL = 'https://earthquake.usgs.gov/fdsnws/event/1/query'


def fetch_region(name, minlat, maxlat, minlon, maxlon, start, end, min_mag):
    """Query USGS FDSN for one geographic box and one time period."""
    params = {
        'format'       : 'csv',
        'starttime'    : start,
        'endtime'      : end,
        'minmagnitude' : min_mag,
        'orderby'      : 'time-asc',
        'limit'        : 20000,
        'minlatitude'  : minlat,
        'maxlatitude'  : maxlat,
        'minlongitude' : minlon,
        'maxlongitude' : maxlon,
    }
    try:
        r = requests.get(USGS_URL, params=params, timeout=180)
        r.raise_for_status()
        df = pd.read_csv(io.StringIO(r.text))
        for col in ['latitude', 'longitude', 'depth', 'mag']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['time'] = pd.to_datetime(df['time'], errors='coerce', utc=True)
        df.dropna(subset=['latitude', 'longitude', 'depth', 'mag', 'time'], inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f'    {name:<24} → {len(df):>5,} events')
        return df
    except Exception as e:
        print(f'    {name:<24} → FAILED: {e}')
        return pd.DataFrame()


if Path(CATALOG_CSV).exists():
    df_catalog = pd.read_csv(CATALOG_CSV)
    df_catalog['time'] = pd.to_datetime(df_catalog['time'], utc=True, errors='coerce')
    print(f'Loaded existing catalog: {len(df_catalog):,} events  ({CATALOG_CSV})')
    print('(Delete catalog_v5_expanded.csv to re-fetch from USGS)')
else:
    print(f'Fetching train period {TRAIN_START} → {TRAIN_END}  (M≥{MIN_MAGNITUDE}) ...')
    tr_frames = []
    for name, la1, la2, lo1, lo2 in SEARCH_REGIONS:
        df = fetch_region(name, la1, la2, lo1, lo2, TRAIN_START, TRAIN_END, MIN_MAGNITUDE)
        if not df.empty:
            tr_frames.append(df)
    df_tr = pd.concat(tr_frames, ignore_index=True) if tr_frames else pd.DataFrame()

    print(f'\nFetching test period {TEST_START} → {TEST_END}  (M≥{MIN_MAGNITUDE}) ...')
    te_frames = []
    for name, la1, la2, lo1, lo2 in SEARCH_REGIONS:
        df = fetch_region(name, la1, la2, lo1, lo2, TEST_START, TEST_END, MIN_MAGNITUDE)
        if not df.empty:
            te_frames.append(df)
    df_te = pd.concat(te_frames, ignore_index=True) if te_frames else pd.DataFrame()

    # Deduplicate by event id (same event may appear in overlapping boxes)
    df_catalog = pd.concat([df_tr, df_te], ignore_index=True)
    if 'id' in df_catalog.columns:
        before = len(df_catalog)
        df_catalog.drop_duplicates(subset='id', inplace=True)
        print(f'\nDeduplication: {before:,} → {len(df_catalog):,} unique events')
    df_catalog.sort_values('time', inplace=True)
    df_catalog.reset_index(drop=True, inplace=True)
    df_catalog.to_csv(CATALOG_CSV, index=False)
    print(f'Saved catalog: {len(df_catalog):,} total events → {CATALOG_CSV}')

# ── Split by date ──────────────────────────────────────────────────────────
train_end_dt = pd.Timestamp(TRAIN_END, tz='UTC')
test_end_dt  = pd.Timestamp(TEST_END,  tz='UTC')

df_train_pool = df_catalog[df_catalog['time'] <  train_end_dt].copy().reset_index(drop=True)
df_test_pool  = df_catalog[(df_catalog['time'] >= train_end_dt) &
                            (df_catalog['time'] <  test_end_dt)].copy().reset_index(drop=True)

print()
print('=' * 60)
print(f'  Train catalog  : {len(df_train_pool):>7,} events  ({TRAIN_START} – {TRAIN_END})')
print(f'  Test  catalog  : {len(df_test_pool):>7,} events  ({TEST_START} – {TEST_END})')
print(f'  Download cap   : {NUM_TRAIN:>7,} train  /  {NUM_TEST:>5,} test')
print('=' * 60)

for pool_name, pool in [('Train', df_train_pool), ('Test', df_test_pool)]:
    print(f'\n{pool_name} magnitude distribution:')
    for lo, hi in [(3.0, 3.5), (3.5, 4.0), (4.0, 5.0), (5.0, 6.0), (6.0, 8.5)]:
        n   = int(((pool['mag'] >= lo) & (pool['mag'] < hi)).sum())
        bar = '█' * min(int(n / max(len(pool), 1) * 50), 50)
        hi_str = str(hi) if hi < 8 else '+'
        print(f'  M{lo}–{hi_str}  {n:>7,}  {bar}')

Loaded existing catalog: 86,881 events  (catalog_v5_expanded.csv)
(Delete catalog_v5_expanded.csv to re-fetch from USGS)

  Train catalog  :  65,185 events  (2000-02-01 – 2020-02-01)
  Test  catalog  :  20,527 events  (2020-02-01 – 2024-01-01)
  Download cap   :  40,000 train  /  5,000 test

Train magnitude distribution:
  M3.0–3.5    1,122  
  M3.5–4.0    6,326  ████
  M4.0–5.0   50,593  ██████████████████████████████████████
  M5.0–6.0    6,535  █████
  M6.0–+      606  

Test magnitude distribution:
  M3.0–3.5        9  
  M3.5–4.0      109  
  M4.0–5.0   18,440  ████████████████████████████████████████████
  M5.0–6.0    1,811  ████
  M6.0–+      158  


## Cell 4 — Download Waveforms (8 threads, overnight-safe)
6 parallel threads, 3-retry logic, 0.3s stagger between threads to avoid IRIS rate limiting. Resumes automatically if interrupted.

In [4]:
# CELL 4 — DOWNLOAD WAVEFORMS FROM IRIS
# Key upgrades vs previous run:
#   NETWORKS   : 10 global/regional networks (unchanged — already optimal)
#   maxradius  : 20° → 25°   (finds stations for Japan/Indonesia/Iran events)
#   MAX_WORKERS: 8  → 16     (faster overnight download)
#   THREAD_DELAY: 0.3 → 0.15 (halved — IRIS allows ~20 req/s per IP)
# Already-downloaded files are skipped automatically — safe to resume.

import io, math, threading, time
import obspy
from obspy.clients.fdsn import Client
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

NETWORKS        = 'IU,II,IN,GE,IA,MN,TW,PS,IC,GT'
CHANNELS        = 'BHZ,HHZ,EHZ'
MAX_WORKERS     = 16              # 16 threads — fast but still polite to IRIS
REQUEST_TIMEOUT = 90
RETRY_ATTEMPTS  = 3
RETRY_DELAY     = 5
THREAD_DELAY    = 0.15            # 0.15s stagger per thread

_lock       = threading.Lock()
_ok_count   = 0
_skip_count = 0
_fail_cache = set()
_target     = 0


def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def _with_retry(fn):
    last_err = None
    for attempt in range(RETRY_ATTEMPTS):
        try:
            return fn()
        except Exception as e:
            last_err = e
            if attempt < RETRY_ATTEMPTS - 1:
                time.sleep(RETRY_DELAY * (attempt + 1))
    raise last_err


def _download_one(row, split, idx):
    global _ok_count, _skip_count

    with _lock:
        if _ok_count >= _target:
            return 'done'

    stem    = f'event_{split}_{idx:05d}'
    mseed_p = Path(WAVEFORM_DIR) / f'{stem}.mseed'
    meta_p  = Path(WAVEFORM_DIR) / f'{stem}_meta.json'

    # already downloaded — count and skip
    if mseed_p.exists() and meta_p.exists():
        with _lock:
            _ok_count += 1
        return 'already'

    time.sleep(THREAD_DELAY)

    try:
        t      = row['time']
        origin = (obspy.UTCDateTime(t.to_pydatetime())
                  if hasattr(t, 'to_pydatetime') else obspy.UTCDateTime(str(t)))
        t_start  = origin - PRE_EVENT_SEC
        t_end    = origin + POST_EVENT_SEC
        eq_lat   = float(row['latitude'])
        eq_lon   = float(row['longitude'])
        eq_dep   = float(row['depth'])
        eq_mag   = float(row['mag'])

        client = Client('IRIS', timeout=REQUEST_TIMEOUT)

        # Wider radius (25°) catches stations for Japan/Indonesia/Iran events
        try:
            inv = _with_retry(lambda: client.get_stations(
                starttime=t_start, endtime=t_end,
                latitude=eq_lat, longitude=eq_lon,
                maxradius=30,
                network=NETWORKS, channel=CHANNELS, level='channel'))
        except Exception:
            with _lock:
                _skip_count += 1
            return 'skipped'

        best      = None
        best_dist = float('inf')
        for net in inv:
            for sta in net:
                key = (net.code, sta.code)
                if key in _fail_cache:
                    continue
                d = haversine(eq_lat, eq_lon, sta.latitude, sta.longitude)
                if d < best_dist:
                    best_dist = d
                    best = (net.code, sta.code, sta.latitude, sta.longitude)

        if best is None:
            with _lock:
                _skip_count += 1
            return 'skipped'

        net_code, sta_code, sta_lat, sta_lon = best

        try:
            st = _with_retry(lambda: client.get_waveforms(
                network=net_code, station=sta_code,
                location='*', channel='BHZ,HHZ,EHZ',
                starttime=t_start, endtime=t_end))
        except Exception:
            with _lock:
                _fail_cache.add((net_code, sta_code))
                _skip_count += 1
            return 'skipped'

        if not st or len(st) == 0:
            with _lock:
                _fail_cache.add((net_code, sta_code))
                _skip_count += 1
            return 'skipped'

        with _lock:
            if _ok_count >= _target:
                return 'done'
            _ok_count += 1

        st.write(str(mseed_p), format='MSEED')
        meta = {
            'eq_lat': eq_lat, 'eq_lon': eq_lon, 'eq_depth_km': eq_dep,
            'magnitude': eq_mag, 'sta_lat': float(sta_lat), 'sta_lon': float(sta_lon),
            'dist_km': float(best_dist), 'origin_time': str(row['time']), 'split': split,
        }
        json.dump(meta, open(str(meta_p), 'w'))
        return 'ok'

    except Exception:
        with _lock:
            _skip_count += 1
        return 'error'


def run_downloads(df, split, target):
    global _ok_count, _skip_count, _fail_cache, _target
    _ok_count = _skip_count = 0
    _fail_cache = set()
    _target = target

    # Count already-done so progress bar starts correctly
    already = sum(
        1 for i in range(len(df))
        if (Path(WAVEFORM_DIR) / f'event_{split}_{i+1:05d}.mseed').exists()
        and (Path(WAVEFORM_DIR) / f'event_{split}_{i+1:05d}_meta.json').exists()
    )
    _ok_count = already

    print(f'[{split.upper()}]  Target={target:,}  Already={already:,}  Workers={MAX_WORKERS}')
    print(f'         Networks={NETWORKS}  maxradius=30°')

    remaining = [(i, row) for i, (_, row) in enumerate(df.iterrows(), 1)
                 if not (Path(WAVEFORM_DIR) / f'event_{split}_{i:05d}.mseed').exists()]

    with tqdm(total=min(target, len(df)), initial=already, unit='evt',
              desc=f'{split} waveforms') as pbar:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {ex.submit(_download_one, row, split, i): i
                    for i, row in remaining
                    if _ok_count < target}
            for fut in as_completed(futs):
                res = fut.result()
                if res in ('ok', 'already'):
                    pbar.update(1)
                if _ok_count >= target:
                    break

    print(f'  Done: {_ok_count:,} downloaded / {_skip_count:,} skipped')


# ── Run downloads ──────────────────────────────────────────────────────────
print(f'Downloading TRAIN waveforms (cap={NUM_TRAIN:,}) ...')
run_downloads(df_train_pool, 'train', NUM_TRAIN)

print()
print(f'Downloading TEST waveforms (cap={NUM_TEST:,}) ...')
run_downloads(df_test_pool, 'test', NUM_TEST)

print()
total_mseed = len(list(Path(WAVEFORM_DIR).glob('*.mseed')))
print(f'Total MiniSEED files on disk: {total_mseed:,}')

[TRAIN]  Target=40,000  Already=18,160  Workers=16
         Networks=IU,II,IN,GE,IA,MN,TW,PS,IC,GT  maxradius=25°


train waveforms:  45%|####5     | 18160/40000 [00:00<?, ?evt/s]

  Done: 26,498 downloaded / 38,687 skipped



[TEST]  Target=5,000  Already=5,000  Workers=16
         Networks=IU,II,IN,GE,IA,MN,TW,PS,IC,GT  maxradius=25°


test waveforms: 100%|##########| 5000/5000 [00:00<?, ?evt/s]

  Done: 5,000 downloaded / 0 skipped

Total MiniSEED files on disk: 31,526


## Cell 5 — Process Waveforms → CSV + Meta JSON
Reads each MSEED, applies bandpass filter, runs STA/LTA labelling, saves CSV + `_meta.json` to `processed_v5/`.

In [5]:
# CELL 5 — PROCESS WAVEFORMS → CSV + META JSON


def normalise_location(dist_km, depth_km, lat, lon):
    return [
        dist_km  / LOC_NORM[0],
        depth_km / LOC_NORM[1],
        (lat  + 90.0)  / LOC_NORM[2],
        (lon  + 180.0) / LOC_NORM[3],
    ]


def process_one(mseed_path):
    stem     = Path(mseed_path).stem
    meta_src = Path(WAVEFORM_DIR)  / f'{stem}_meta.json'
    csv_out  = Path(PROCESSED_DIR) / f'{stem}.csv'
    meta_out = Path(PROCESSED_DIR) / f'{stem}_meta.json'

    if csv_out.exists() and meta_out.exists():
        return 'already'
    if not meta_src.exists():
        return 'no_meta'

    try:
        meta = json.load(open(str(meta_src)))
    except Exception:
        return 'bad_meta'

    try:
        st = obspy.read(str(mseed_path))
    except Exception:
        return 'bad_mseed'

    if len(st) == 0:
        return 'empty'

    tr = st[0].copy()
    sr = tr.stats.sampling_rate
    if sr < 10:
        return 'low_sr'

    tr.detrend('demean')
    tr.detrend('linear')
    tr.taper(max_percentage=0.05, type='cosine')
    freqmax = min(FREQMAX, sr / 2.0 - 0.5)
    if freqmax <= FREQMIN:
        return 'bad_freq'
    tr.filter('bandpass', freqmin=FREQMIN, freqmax=freqmax, corners=4, zerophase=True)

    data = tr.data.astype(np.float64)
    std  = data.std()
    if std < 1e-10:
        return 'flat'
    data = (data - data.mean()) / std

    sta_n = int(STA_SEC * sr)
    lta_n = int(LTA_SEC * sr)
    if len(data) < lta_n * 2:
        return 'too_short'
    try:
        cft    = recursive_sta_lta(data, sta_n, lta_n)
        onsets = trigger_onset(cft, THR_ON, THR_OFF)
    except Exception:
        return 'sta_lta_fail'

    if len(onsets) == 0:
        return 'no_onset'

    labels  = np.zeros(len(data), dtype=np.int8)
    p_start = onsets[0][0]
    if len(onsets) >= 3:
        labels[p_start          : onsets[1][0]] = 1
        labels[onsets[1][0]     : onsets[2][0]] = 2
        labels[onsets[2][0]    :              ] = 3
    elif len(onsets) == 2:
        labels[p_start : onsets[1][0]] = 1
        labels[onsets[1][0]:         ] = 2
    else:
        labels[p_start:] = 1

    if (labels == 1).sum() / len(labels) < MIN_P_FRACTION:
        return 'no_p_wave'

    loc = normalise_location(meta['dist_km'], meta['eq_depth_km'],
                              meta['eq_lat'], meta['eq_lon'])
    n   = len(data)
    df  = pd.DataFrame({
        'time_sec'  : np.arange(n) / sr,
        'amplitude' : data,
        'label'     : labels,
        'dist_km_n' : np.full(n, loc[0], dtype=np.float32),
        'depth_km_n': np.full(n, loc[1], dtype=np.float32),
        'lat_n'     : np.full(n, loc[2], dtype=np.float32),
        'lon_n'     : np.full(n, loc[3], dtype=np.float32),
        'split'     : meta.get('split', 'train'),
    })
    df.to_csv(str(csv_out), index=False)
    shutil.copy(str(meta_src), str(meta_out))
    return 'ok'


mseed_files = sorted(Path(WAVEFORM_DIR).glob('*.mseed'))
print(f'Processing {len(mseed_files)} MSEED files ...')

counts = {}
for fp in mseed_files:
    r = process_one(str(fp))
    counts[r] = counts.get(r, 0) + 1
    if sum(counts.values()) % 500 == 0:
        print(f'  {sum(counts.values())} done — {counts}')

print()
print('Processing complete:')
for k, v in sorted(counts.items()):
    print(f'  {k:20s} : {v:,}')
ok_count = counts.get('ok', 0) + counts.get('already', 0)
print(f'\nTotal usable events in {PROCESSED_DIR}: {ok_count:,}')


Processing 31526 MSEED files ...


  500 done — {'already': 473, 'too_short': 7, 'no_onset': 20}
  1000 done — {'already': 951, 'too_short': 7, 'no_onset': 42}


  1500 done — {'already': 1417, 'too_short': 8, 'no_onset': 75}
  2000 done — {'already': 1904, 'too_short': 9, 'no_onset': 87}
  2500 done — {'already': 2385, 'too_short': 10, 'no_onset': 105}
  3000 done — {'already': 2872, 'too_short': 10, 'no_onset': 118}


  3500 done — {'already': 3355, 'too_short': 10, 'no_onset': 135}
  4000 done — {'already': 3828, 'too_short': 10, 'no_onset': 162}


  4500 done — {'already': 4303, 'too_short': 10, 'no_onset': 187}
  5000 done — {'already': 4784, 'too_short': 10, 'no_onset': 206}


...

  29500 done — {'already': 20640, 'too_short': 11, 'no_onset': 1985, 'ok': 6857, 'flat': 7}


  30000 done — {'already': 20988, 'too_short': 11, 'no_onset': 2042, 'ok': 6952, 'flat': 7}


  30500 done — {'already': 21186, 'too_short': 11, 'no_onset': 2084, 'ok': 7212, 'flat': 7}


  31000 done — {'already': 21468, 'too_short': 11, 'no_onset': 2124, 'ok': 7390, 'flat': 7}


  31500 done — {'already': 21700, 'too_short': 13, 'no_onset': 2163, 'ok': 7617, 'flat': 7}

Processing complete:
  already              : 21,720
  flat                 : 7
  no_onset             : 2,169
  ok                   : 7,617
  too_short            : 13

Total usable events in processed_v5: 29,337


## Cell 6 — Migrate Existing v4 Data (Optional)
Copies `_meta.json` files from `waveforms_temporal/` → `processed_temporal/` so v5 training can reuse existing v4 data without re-downloading.

In [6]:
# CELL 6 — MIGRATE v4 DATA (optional — only needed if reusing processed_temporal/)

V4_PROC = Path('processed_temporal')
V4_WAVE = Path('waveforms_temporal')

if not V4_PROC.exists() or not V4_WAVE.exists():
    print('No v4 directories found — skipping.')
else:
    csv_files = sorted(V4_PROC.glob('*.csv'))
    copied = missing = already = 0
    for csv_p in csv_files:
        meta_src = V4_WAVE / f'{csv_p.stem}_meta.json'
        meta_dst = V4_PROC / f'{csv_p.stem}_meta.json'
        if meta_dst.exists():
            already += 1
            continue
        if meta_src.exists():
            shutil.copy(str(meta_src), str(meta_dst))
            copied += 1
        else:
            missing += 1
    print(f'Migration done:  copied={copied:,}  already={already:,}  missing={missing:,}')
    print('Set PROCESSED_DIR = "processed_temporal" in training_v5.ipynb to use this data.')


No v4 directories found — skipping.


## Cell 7 — Verify Output
Sanity check: confirms CSV + meta JSON exist and magnitude is present.

In [7]:
# CELL 7 — VERIFY OUTPUT

import random

check_dir  = Path(PROCESSED_DIR)
csv_files  = sorted(check_dir.glob('*.csv'))
meta_files = sorted(check_dir.glob('*_meta.json'))

print(f'CSV files  : {len(csv_files):,}')
print(f'Meta JSON  : {len(meta_files):,}')

if len(csv_files) == 0:
    print('WARNING: No CSV files found. Run Cell 5 first.')
else:
    sample = random.sample(csv_files, min(5, len(csv_files)))
    print()
    for p in sample:
        meta_p = p.parent / f'{p.stem}_meta.json'
        try:
            df_s   = pd.read_csv(p)
            meta   = json.load(open(str(meta_p)))
            mag    = meta.get('magnitude', 'MISSING')
            p_pct  = round(100 * (df_s['label'] == 1).sum() / max(len(df_s), 1), 1)
            split  = df_s['split'].iloc[0]
            print(f'  {p.name}')
            print(f'    rows={len(df_s):,}  split={split}  magnitude={mag}  P%={p_pct}%')
        except Exception as e:
            print(f'  {p.name} — ERROR: {e}')


CSV files  : 29,337
Meta JSON  : 29,337

  event_train_25698.csv
    rows=3,000  split=train  magnitude=4.3  P%=25.9%
  event_train_05236.csv
    rows=3,000  split=train  magnitude=3.5  P%=26.0%
  event_train_25840.csv
    rows=3,000  split=train  magnitude=4.2  P%=24.2%
  event_train_35714.csv
    rows=3,000  split=train  magnitude=4.5  P%=93.3%
  event_train_28687.csv
    rows=3,000  split=train  magnitude=3.4  P%=8.5%
